In [80]:
import sqlite3
import pandas as pd

In [81]:
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

In [82]:
cursor.execute("""
CREATE TABLE customers (
    customer_id INTEGER,
    name TEXT,
    city TEXT
)
""")

cursor.execute("""
CREATE TABLE orders (
    order_id INTEGER,
    customer_id INTEGER,
    amount REAL
)
""")

In [83]:
cursor.execute("""
               INSERT INTO customers VALUES
               (1, 'Jan', 'Kraków'),
                (2, 'Anna', 'Warszawa'),
                (3, 'Piotr', 'Gdańsk'),
                (4, 'Maria', 'Poznań'),
                (5, 'Tomasz', 'Wrocław'),
                (6, 'Karolina', 'Lublin')
                """)
cursor.execute("""
                INSERT INTO orders VALUES
                (101, 1, 250),
                (102, 2, 800),
                (103, 1, 150),
                (104, 4, 1200),
                (105, 5, 400)
""")
conn.commit()

In [84]:
pd.read_sql_query("SELECT * FROM customers", conn)

,customer_id,name,city
0,1,Jan,Kraków
1,2,Anna,Warszawa
2,3,Piotr,Gdańsk
3,4,Maria,Poznań
4,5,Tomasz,Wrocław
5,6,Karolina,Lublin


In [85]:
pd.read_sql_query("SELECT * FROM orders", conn)

,order_id,customer_id,amount
0,101,1,250.0
1,102,2,800.0
2,103,1,150.0
3,104,4,1200.0
4,105,5,400.0


In [86]:
pd.read_sql_query("""
SELECT
    
    customers.name,
    customers.city,

    orders.amount
FROM customers
INNER JOIN orders
ON customers.customer_id = orders.customer_id
ORDER BY orders.amount DESC
""", conn)

,name,city,amount
0,Maria,Poznań,1200.0
1,Anna,Warszawa,800.0
2,Tomasz,Wrocław,400.0
3,Jan,Kraków,250.0
4,Jan,Kraków,150.0


In [87]:
pd.read_sql_query(""" SELECT
    customers.customer_id,
    customers.name,
    customers.city,
    orders.order_id,
    orders.amount
FROM customers
LEFT JOIN orders
ON customers.customer_id = orders.customer_id
""", conn)

,customer_id,name,city,order_id,amount
0,1,Jan,Kraków,101.0,250.0
1,1,Jan,Kraków,103.0,150.0
2,2,Anna,Warszawa,102.0,800.0
3,3,Piotr,Gdańsk,NaN,NaN
4,4,Maria,Poznań,104.0,1200.0
5,5,Tomasz,Wrocław,105.0,400.0
6,6,Karolina,Lublin,NaN,NaN


In [88]:
pd.read_sql_query("""SELECT
    customers.customer_id,
    customers.name,
    customers.city,
    orders.order_id,
    COALESCE(orders.amount, 0) AS amount
FROM customers
LEFT JOIN orders
ON customers.customer_id = orders.customer_id
""", conn)


,customer_id,name,city,order_id,amount
0,1,Jan,Kraków,101.0,250.0
1,1,Jan,Kraków,103.0,150.0
2,2,Anna,Warszawa,102.0,800.0
3,3,Piotr,Gdańsk,NaN,0.0
4,4,Maria,Poznań,104.0,1200.0
5,5,Tomasz,Wrocław,105.0,400.0
6,6,Karolina,Lublin,NaN,0.0


In [89]:
pd.read_sql_query("""SELECT
    customers.customer_id,
    customers.name,
    orders.order_id,
    orders.amount
FROM customers
LEFT JOIN orders
ON customers.customer_id = orders.customer_id
WHERE orders.order_id IS NULL
""", conn)

,customer_id,name,order_id,amount
0,3,Piotr,None,None
1,6,Karolina,None,None


In [90]:
pd.read_sql_query("""SELECT customer_id, 
COUNT(*) AS liczba_zamowien
FROM orders     
GROUP BY customer_id
""", conn)


,customer_id,liczba_zamowien
0,1,2
1,2,1
2,4,1
3,5,1


In [91]:
pd.read_sql_query("""
SELECT *
FROM orders
""", conn)

,order_id,customer_id,amount
0,101,1,250.0
1,102,2,800.0
2,103,1,150.0
3,104,4,1200.0
4,105,5,400.0


In [92]:
pd.read_sql_query("""SELECT
    customer_id,
    COUNT(*) AS liczba_zamowien,
    SUM(amount) AS laczna_kwota,
    AVG(amount) AS srednia_kwota
FROM orders
GROUP BY customer_id
""", conn)

,customer_id,liczba_zamowien,laczna_kwota,srednia_kwota
0,1,2,400.0,200.0
1,2,1,800.0,800.0
2,4,1,1200.0,1200.0
3,5,1,400.0,400.0


In [93]:
cursor.execute("""
CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    name TEXT,
    category TEXT,
    price REAL,
    stock INTEGER
)
""")
cursor.execute("""
INSERT INTO products VALUES
(1, 'Dell XPS 13', 'Laptop', 5200, 5),
(2, 'Lenovo ThinkPad', 'Laptop', 4800, 7),
(3, 'LG UltraGear', 'Monitor', 1500, 12),
(4, 'Samsung Odyssey', 'Monitor', 2200, 6),
(5, 'Logitech MX Keys', 'Klawiatura', 450, 20),
(6, 'Keychron K8', 'Klawiatura', 420, 15),
(7, 'Logitech MX Master 3', 'Mysz', 390, 18),
(8, 'Razer DeathAdder', 'Mysz', 280, 25),
(9, 'Sony WH-1000XM5', 'Słuchawki', 1700, 8),
(10, 'HyperX Cloud II', 'Słuchawki', 450, 14)
""")

conn.commit()

In [94]:
pd.read_sql_query("""
SELECT *
FROM products
""", conn)

,product_id,name,category,price,stock
0,1,Dell XPS 13,Laptop,5200.0,5
1,2,Lenovo ThinkPad,Laptop,4800.0,7
2,3,LG UltraGear,Monitor,1500.0,12
3,4,Samsung Odyssey,Monitor,2200.0,6
4,5,Logitech MX Keys,Klawiatura,450.0,20
5,6,Keychron K8,Klawiatura,420.0,15
6,7,Logitech MX Master 3,Mysz,390.0,18
7,8,Razer DeathAdder,Mysz,280.0,25
8,9,Sony WH-1000XM5,Słuchawki,1700.0,8
9,10,HyperX Cloud II,Słuchawki,450.0,14


In [95]:
pd.read_sql_query("""SELECT
    category,
    SUM(stock) AS liczba_sztuk
FROM products
GROUP BY category
""", conn)


,category,liczba_sztuk
0,Klawiatura,35
1,Laptop,12
2,Monitor,18
3,Mysz,43
4,Słuchawki,22


In [96]:
pd.read_sql_query("""SELECT
    COUNT(*) AS liczba_produktow
FROM products
;
""", conn)


,liczba_produktow
0,10


In [97]:
pd.read_sql_query(""" SELECT
    SUM(stock * price)
FROM products
""", conn)

,SUM(stock * price)
0,140020.0


In [98]:
pd.read_sql_query(""" SELECT category, 
AVG(price) AS SREDNIA_CENA
FROM products
GROUP BY category
""", conn)

,category,SREDNIA_CENA
0,Klawiatura,435.0
1,Laptop,5000.0
2,Monitor,1850.0
3,Mysz,335.0
4,Słuchawki,1075.0


zad 8


In [99]:
import numpy as np
import time

In [100]:
cursor.execute("""
DROP TABLE IF EXISTS products_big;
""")

cursor.execute("""
CREATE TABLE products_big (
    product_id INTEGER PRIMARY KEY,
    category TEXT,
    price REAL,
    stock INTEGER
)
""")

In [101]:
categories = np.random.choice(
    ["Laptop", "Monitor", "Mysz", "Klawiatura"],
    size=10000
)

prices = np.random.randint(100, 5000, size=10000)

stocks = np.random.randint(1, 50, size=10000)

In [102]:
data = list(zip(categories, prices, stocks))

In [103]:
cursor.executemany("""
INSERT INTO products_big(category, price, stock)
VALUES (?, ?, ?)
""", data)

conn.commit()

In [104]:
prices = np.random.randint(100, 5000, size=10000)
stocks = np.random.randint(1, 50, size=10000)

data = list(zip(categories, prices, stocks))

cursor.executemany("""
INSERT INTO products_big (category, price, stock)
VALUES (?, ?, ?)
""", data)

conn.commit()

In [105]:
query = """
SELECT *
FROM products_big
WHERE category = 'Laptop'
"""

In [106]:
times = []

for _ in range(5):
    start = time.perf_counter()
    pd.read_sql_query(query, conn)
    end = time.perf_counter()
    times.append(end - start)

avg_without = sum(times) / len(times)

print("Średni czas bez indeksu:", avg_without)

Średni czas bez indeksu: 0.01235717481467873


In [112]:
plan = pd.read_sql_query("""
EXPLAIN QUERY PLAN
SELECT *
FROM products_big
WHERE category = 'Laptop'
""", conn)

print(plan)

   id  parent  notused                                             detail
0   3       0       62  SEARCH products_big USING INDEX idx_category (...


In [111]:
cursor.execute("""
CREATE INDEX idx_category
ON products_big(category)
""")

conn.commit()

OperationalError: index idx_category already exists

In [109]:
print(f"Czas bez indeksu: {avg_without:.8f} s")
print(f"Czas z indeksem : {avg_with:.8f} s")
print(f"Przyspieszenie : {avg_without / avg_with:.2f}x")

Czas bez indeksu: 0.01235717 s


NameError: name 'avg_with' is not defined

In [113]:
times = []

for _ in range(5):
    start = time.perf_counter()
    pd.read_sql_query(query, conn)
    end = time.perf_counter()
    times.append(end - start)

avg_with = sum(times) / len(times)

print("Średni czas z indeksem:", avg_with)

Średni czas z indeksem: 0.013694408396258951


In [114]:
plan = pd.read_sql_query("""
EXPLAIN QUERY PLAN
SELECT *
FROM products_big
WHERE category = 'Laptop'
""", conn)

print(plan)

   id  parent  notused                                             detail
0   3       0       62  SEARCH products_big USING INDEX idx_category (...


In [115]:
print(f"Czas bez indeksu: {avg_without:.8f} s")
print(f"Czas z indeksem: {avg_with:.8f} s")
print(f"Przyspieszenie: {avg_without / avg_with:.2f}x")

Czas bez indeksu: 0.01235717 s
Czas z indeksem: 0.01369441 s
Przyspieszenie: 0.90x


zad 18. 


In [116]:
cursor.execute("""
CREATE TABLE employees (
    employee_id INTEGER,
    name TEXT,
    manager_id INTEGER
)
""")

In [117]:
cursor.executemany("""
INSERT INTO employees (employee_id, name, manager_id)
VALUES (?, ?, ?)
""", [
    (1, "Anna", None),
    (2, "Jan", 1),
    (3, "Piotr", 1),
    (4, "Kasia", 2),
    (5, "Tomek", 2),
    (6, "Marek", 3)
])

conn.commit()

In [118]:
pd.read_sql_query("""
SELECT *
FROM employees
""", conn)

,employee_id,name,manager_id
0,1,Anna,NaN
1,2,Jan,1.0
2,3,Piotr,1.0
3,4,Kasia,2.0
4,5,Tomek,2.0
5,6,Marek,3.0


In [119]:
pd.read_sql_query("""
SELECT
    e1.name AS employee,
    e2.name AS manager
FROM employees e1
LEFT JOIN employees e2
ON e1.manager_id = e2.employee_id
""", conn)

,employee,manager
0,Anna,NaN
1,Jan,Anna
2,Piotr,Anna
3,Kasia,Jan
4,Tomek,Jan
5,Marek,Piotr


In [120]:
pd.read_sql_query("""
SELECT
    e1.name AS employee
FROM employees e1
LEFT JOIN employees e2
ON e1.manager_id = e2.employee_id
WHERE e1.manager_id IS NULL
""", conn)

,employee
0,Anna


In [122]:
pd.read_sql_query("""
SELECT
    e2.name AS manager,
    COUNT(*) AS liczba_podwladnych
FROM employees e1
LEFT JOIN employees e2
ON e1.manager_id = e2.employee_id
WHERE e2.name IS NOT NULL
GROUP BY e2.name
""", conn)

,manager,liczba_podwladnych
0,Anna,2
1,Jan,2
2,Piotr,1
